In [1]:
import os, sys, time, random, csv
from pathlib import Path
from datetime import datetime

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from consultas import CONSULTAS

ARCHIVO = Path("../results/mediciones.csv")
NUCLEOS = ["1", "*"]          # local[1] y local[*]

def crear_sesion(nucleos):
    return (SparkSession.builder
            .master(f"local[{nucleos}]")
            .appName("medicion")
            .config("spark.driver.memory", "8g")
            .config("spark.python.worker.faulthandler.enabled", "true")
            .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
            .config("spark.sql.optimizer.excludedRules",
                    "org.apache.spark.sql.catalyst.optimizer.InferFiltersFromConstraints")
            .getOrCreate())

def ejecutar(resultado):
    # Obliga a Spark a calcular el resultado completo, sin traerlo a la memoria del notebook
    if hasattr(resultado, "write"):
        resultado.write.format("noop").mode("overwrite").save()   # DataFrame, SQL y UDF
    else:
        resultado.foreach(lambda _: None)                         # RDD

def ya_medidas():
    if not ARCHIVO.exists():
        return set()
    with open(ARCHIVO, encoding="utf-8") as f:
        return {(r["nucleos"], r["ronda"], r["consulta"], r["variante"])
                for r in csv.DictReader(f) if r["estado"] == "OK"}

def medir(rondas):
    hechas = ya_medidas()
    nuevo = not ARCHIVO.exists()
    with open(ARCHIVO, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["fecha_hora", "nucleos", "ronda", "consulta", "variante", "segundos", "estado"])
        for nucleos in NUCLEOS:
            spark = crear_sesion(nucleos)
            df = spark.read.parquet("../data/secop.parquet")
            print(f"\n=== Sesión local[{nucleos}]: calentamiento ===")
            for v in ["rdd", "df", "sql", "udf"]:
                ejecutar(CONSULTAS["Q1"][v](df))
            for ronda in rondas:
                combos = [(q, v) for q in CONSULTAS for v in CONSULTAS[q]]
                random.Random(f"{nucleos}-{ronda}").shuffle(combos)
                for q, v in combos:
                    if (nucleos, str(ronda), q, v) in hechas:
                        continue
                    t0 = time.perf_counter()
                    try:
                        ejecutar(CONSULTAS[q][v](df))
                        estado = "OK"
                    except Exception as e:
                        estado = f"ERROR {type(e).__name__}"
                    seg = time.perf_counter() - t0
                    w.writerow([datetime.now().isoformat(timespec="seconds"),
                                nucleos, ronda, q, v, f"{seg:.3f}", estado])
                    f.flush()
                    print(f"local[{nucleos}] ronda {ronda}  {q} {v:>3}: {seg:7.1f} s  {estado}")
            spark.stop()
    print("\nListo.")

In [2]:
inicio = time.perf_counter()
medir(rondas=[0])
print(f"\nEl piloto tardó {(time.perf_counter() - inicio) / 60:.1f} minutos")


=== Sesión local[1]: calentamiento ===

=== Sesión local[*]: calentamiento ===

Listo.

El piloto tardó 1.7 minutos


In [3]:
import pandas as pd

m = pd.read_csv(ARCHIVO, dtype={"nucleos": str})
p = m[(m.ronda == 0) & (m.estado == "OK")]

print("Errores en el piloto:", (m[m.ronda == 0].estado != "OK").sum())
print("\nSegundos por combinación:")
print(p.pivot_table(index=["consulta", "variante"], columns="nucleos", values="segundos").round(1))

horas = 3 * p.segundos.sum() / 3600
print(f"\nEstimación para las rondas 1, 2 y 3: {horas:.1f} horas")
print("Regla: si pasa de 8 horas, se quita la Q4.")

Errores en el piloto: 0

Segundos por combinación:
nucleos                *     1
consulta variante             
Q1       df          0.9   1.3
         rdd        33.3  12.5
         sql         0.9   1.0
         udf        16.5   7.0
Q2       df          0.4   1.5
         rdd        19.9  13.5
         sql         0.8   1.1
         udf        16.1   2.6
Q3       df          3.0   2.4
         rdd        32.7  24.7
         sql         2.5   3.3
         udf        22.6   7.6
Q4       df          9.0   5.3
         rdd        77.1  29.6
         sql         6.9   6.1
         udf        28.5  11.9
Q5       df          3.2   2.9
         rdd       117.6  26.0
         sql         4.7   2.4
         udf        18.6   8.0
Q6       df          4.5   6.3
         rdd        33.4  16.1
         sql         3.9   6.9
         udf        18.6  12.4

Estimación para las rondas 1, 2 y 3: 0.6 horas
Regla: si pasa de 8 horas, se quita la Q4.


In [2]:
import psutil, threading

def es_python(p):
    return (p.info["name"] or "").lower().startswith("python")

class Monitor:
    """Toma una muestra del estado del equipo cada medio segundo mientras corre una consulta."""
    def __init__(self):
        self.muestras, self._parar = [], threading.Event()
        self.defender = [p for p in psutil.process_iter(["name"])
                         if (p.info["name"] or "").lower() == "msmpeng.exe"]
        self.acceso_defender = True
        for p in self.defender:
            try:
                p.cpu_percent(None)
            except psutil.Error:
                self.acceso_defender = False
        self.python_base = sum(1 for p in psutil.process_iter(["name"]) if es_python(p))

    def _tomar(self):
        psutil.cpu_percent(None)
        disco0, swap0 = psutil.disk_io_counters(), psutil.swap_memory().used
        while not self._parar.is_set():
            time.sleep(0.5)
            cpu_def = None
            if self.acceso_defender and self.defender:
                try:
                    cpu_def = sum(p.cpu_percent(None) for p in self.defender) / psutil.cpu_count()
                except psutil.Error:
                    self.acceso_defender = False
            self.muestras.append({
                "cpu": psutil.cpu_percent(None),
                "memoria": psutil.virtual_memory().percent,
                "python": sum(1 for p in psutil.process_iter(["name"]) if es_python(p)) - self.python_base,
                "defender": cpu_def,
            })
        disco1 = psutil.disk_io_counters()
        self.disco_mb = (disco1.read_bytes + disco1.write_bytes
                         - disco0.read_bytes - disco0.write_bytes) / 1e6
        self.swap_mb = (psutil.swap_memory().used - swap0) / 1e6

    def __enter__(self):
        self._hilo = threading.Thread(target=self._tomar, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *args):
        self._parar.set()
        self._hilo.join()

    def resumen(self):
        m = self.muestras
        prom = lambda k: sum(x[k] for x in m) / len(m)
        maxi = lambda k: max(x[k] for x in m)
        defender = [x["defender"] for x in m if x["defender"] is not None]
        print(f"  CPU total:          promedio {prom('cpu'):.0f} %, máximo {maxi('cpu'):.0f} %")
        print(f"  Memoria RAM:        promedio {prom('memoria'):.0f} %, máximo {maxi('memoria'):.0f} %")
        print(f"  Memoria virtual:    {self.swap_mb:+,.0f} MB durante la consulta")
        print(f"  Disco leído+escrito:{self.disco_mb:,.0f} MB")
        print(f"  Procesos Python:    máximo {maxi('python')} a la vez (además del notebook)")
        if defender:
            print(f"  Antivirus:          promedio {sum(defender)/len(defender):.1f} %, máximo {max(defender):.1f} % de CPU")
        else:
            print("  Antivirus:          Windows no permite leer su consumo")

for nucleos in ["1", "*"]:
    spark = crear_sesion(nucleos)
    df = spark.read.parquet("../data/secop.parquet")
    print(f"local[{nucleos}]: particiones de lectura = {df.rdd.getNumPartitions()}")
    with Monitor() as mon:
        t0 = time.perf_counter()
        ejecutar(CONSULTAS["Q5"]["rdd"](df))
        seg = time.perf_counter() - t0
    print(f"local[{nucleos}]: Q5 rdd = {seg:.1f} s")
    mon.resumen()
    print()
    spark.stop()

local[1]: particiones de lectura = 2
local[1]: Q5 rdd = 28.0 s
  CPU total:          promedio 13 %, máximo 38 %
  Memoria RAM:        promedio 56 %, máximo 57 %
  Memoria virtual:    +0 MB durante la consulta
  Disco leído+escrito:8 MB
  Procesos Python:    máximo 4 a la vez (además del notebook)
  Antivirus:          promedio 0.8 %, máximo 2.5 % de CPU

local[*]: particiones de lectura = 17
local[*]: Q5 rdd = 119.3 s
  CPU total:          promedio 14 %, máximo 55 %
  Memoria RAM:        promedio 57 %, máximo 60 %
  Memoria virtual:    -4 MB durante la consulta
  Disco leído+escrito:60 MB
  Procesos Python:    máximo 31 a la vez (además del notebook)
  Antivirus:          promedio 1.6 %, máximo 7.9 % de CPU



In [3]:
for nucleos in ["1", "*"]:
    spark = crear_sesion(nucleos)
    sc = spark.sparkContext
    print(f"local[{nucleos}]:")
    for intento in range(1, 4):
        with Monitor() as mon:
            t0 = time.perf_counter()
            sc.parallelize(range(16), 16).map(lambda x: x).count()
            seg = time.perf_counter() - t0
        maximo_python = max(x["python"] for x in mon.muestras) if mon.muestras else 0
        print(f"  intento {intento}: {seg:5.1f} s   procesos Python: {maximo_python}")
    spark.stop()
    print()

local[1]:
  intento 1:  15.0 s   procesos Python: 4
  intento 2:  14.2 s   procesos Python: 4
  intento 3:  14.2 s   procesos Python: 4

local[*]:
  intento 1:  12.9 s   procesos Python: 4
  intento 2:  13.0 s   procesos Python: 4
  intento 3:  12.9 s   procesos Python: 4

